# Day 20: Containerization — Docker & NVIDIA NIMs
> *Inference Engineering* — Chapter 7.1 | Philip Kiely, Baseten Books 2026

**Layer:** Infrastructure | **Prerequisite:** Day 19 (MIG)


## Concept Overview

Containerization is the standard deployment unit for inference services. Docker containers encapsulate the model, runtime, and dependencies into a reproducible image. NVIDIA Container Toolkit (nvidia-docker) passes GPUs to containers with the correct drivers. NVIDIA NIMs (NIM Inference Microservices) are pre-built, optimized containers for NVIDIA-supported models — each NIM includes the model weights, TensorRT-LLM engine, and an OpenAI-compatible API server. NIMs abstract away the runtime complexity: you pull an image, run it, and get an API endpoint.


In [ ]:
import subprocess, os, json
print('Container and NIM concepts demonstration')
print('(No Docker daemon required for code analysis)')


## 1. Dockerfile Structure for Inference Servers

A well-structured inference Dockerfile has four layers: base image, system deps, Python deps, and model/code.


In [ ]:
dockerfile_content = '''
# Stage 1: Base CUDA image
FROM nvcr.io/nvidia/cuda:12.4.1-cudnn9-runtime-ubuntu22.04

# System dependencies
RUN apt-get update && apt-get install -y \\
    python3.10 python3-pip git curl \\
    && rm -rf /var/lib/apt/lists/*

# Python dependencies (cached layer)
COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

# Application code
COPY . /app
WORKDIR /app

# Model weights (use volume or download at runtime, not in image)
ENV MODEL_PATH=/models/llama-3-8b
ENV MAX_MODEL_LEN=8192
ENV TENSOR_PARALLEL_SIZE=1

# Health check
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Expose port
EXPOSE 8000

# Run vLLM server
CMD ["python3", "-m", "vllm.entrypoints.openai.api_server", \\
     "--model", "/models/llama-3-8b", \\
     "--port", "8000", \\
     "--tensor-parallel-size", "1"]
'''
print('Sample inference Dockerfile:')
print(dockerfile_content)

# Layer size analysis
print('Typical layer sizes:')
layers = [
    ('Base CUDA 12.4 runtime', '1.2 GB', 'Pulled from NVIDIA NGC, cached across deployments'),
    ('Python + system deps',   '0.5 GB', 'Changes rarely — cache-friendly'),
    ('Python requirements',    '0.8 GB', 'vLLM, transformers, etc.'),
    ('Application code',       '0.01 GB','Changes frequently — keep thin'),
    ('Model weights (volume)', '16 GB',  'NOT in image — mounted at runtime'),
]
for name, size, note in layers:
    print(f'  {name:<30} {size:<10} {note}')


## 2. NVIDIA Container Toolkit

nvidia-docker passes GPUs to containers. The key flags: `--gpus all` or `--gpus device=0,1`.


In [ ]:
print('NVIDIA Container Toolkit usage:')
print()
commands = [
    ('Run with all GPUs',
     'docker run --gpus all -p 8000:8000 my-inference-server'),
    ('Run with specific GPU',
     'docker run --gpus device=0 -p 8000:8000 my-inference-server'),
    ('Run with MIG instance',
     'docker run --gpus \"device=MIG-GPU-xxx:0\" -p 8000:8000 my-inference-server'),
    ('With model weights volume',
     'docker run --gpus all -v /models:/models -p 8000:8000 my-server'),
    ('With shared memory for vLLM',
     'docker run --gpus all --shm-size=10g -p 8000:8000 vllm-server'),
]
for desc, cmd in commands:
    print(f'  {desc}:')
    print(f'    {cmd}')
    print()

print('Environment variables for model config:')
env_vars = [
    ('MODEL_NAME',              'Model identifier (HuggingFace or local path)'),
    ('MAX_MODEL_LEN',           'Maximum context length'),
    ('TENSOR_PARALLEL_SIZE',    'Number of GPUs for tensor parallelism'),
    ('GPU_MEMORY_UTILIZATION',  'Fraction of GPU memory to use (e.g., 0.9)'),
    ('QUANTIZATION',            'Quantization method (awq, gptq, int8)'),
]
for var, desc in env_vars:
    print(f'  {var:<30} {desc}')


## 3. NVIDIA NIMs

NIMs are pre-optimized inference containers from NVIDIA NGC. They include compiled TensorRT-LLM engines and expose an OpenAI-compatible API.


In [ ]:
print('NVIDIA NIM: What\'s inside')
print()
nim_contents = [
    'Compiled TensorRT-LLM engine (GPU-arch specific)',
    'OpenAI-compatible REST API (/v1/completions, /v1/chat/completions)',
    'Health check endpoint (/health)',
    'Metrics endpoint (/metrics) — Prometheus format',
    'Batching and scheduling logic',
    'Model weights (baked in or fetched from NGC)',
]
for item in nim_contents:
    print(f'  + {item}')

print()
print('Running a Llama-3-8B NIM:')
print('''
  # Pull and run (requires NGC API key)
  export NGC_API_KEY=<your_key>
  docker login nvcr.io --username \\$oauthtoken --password $NGC_API_KEY

  docker run --gpus all \\
    -e NGC_API_KEY=$NGC_API_KEY \\
    -v /models:/opt/nim/.cache \\
    -p 8000:8000 \\
    nvcr.io/nim/meta/llama-3.1-8b-instruct:latest

  # Test
  curl http://localhost:8000/v1/chat/completions \\
    -H "Content-Type: application/json" \\
    -d \'{"model": "meta/llama-3.1-8b-instruct",
          "messages": [{"role": "user", "content": "Hello!"}]}\'\n''')

print('NIM vs custom vLLM container tradeoffs:')
for aspect, nim_val, custom_val in [
    ('Setup time',     'Minutes',              'Hours'),
    ('Optimization',   'NVIDIA-tuned TRT-LLM', 'Manual tuning needed'),
    ('Flexibility',    'NVIDIA-supported models', 'Any model'),
    ('Custom kernels', 'No',                   'Yes'),
    ('Debugging',      'Black box',            'Full access'),
    ('Cost',           'NGC subscription',     'Free'),
]:
    print(f'  {aspect:<20} {nim_val:<30} {custom_val}')


## Experiments: Try These

1. **Build a vLLM Docker image**: Write a Dockerfile, build it, and run a Llama-3.2-1B inference server on spark-01. Test with curl.
2. **Multi-stage build**: Optimize the Dockerfile with multi-stage build to separate build dependencies from runtime. Measure image size reduction.
3. **NIM trial**: Pull a NIM container from NGC and compare TTFT against your hand-built vLLM container on the same GPU.


## Key Takeaways

- Docker containers provide reproducible inference deployments; model weights should be mounted as volumes (not baked into the image).
- NVIDIA Container Toolkit passes GPUs to containers; `--gpus` controls device assignment including MIG instance routing.
- NIMs are pre-optimized TensorRT-LLM containers with OpenAI-compatible APIs — the fastest path to production for NVIDIA-supported models.
- Layer caching matters: put rarely-changing layers (base image, Python deps) at the top; frequently-changing code at the bottom.

## References
- *Inference Engineering* Ch 7.1 — Philip Kiely, Baseten Books 2026
- NVIDIA Container Toolkit documentation
- NVIDIA NIM documentation (NGC)
